In [ ]:
!apt-get install -y poppler-utils
!pip install opencv-python numpy pdf2image img2pdf

In [ ]:
!apt-get install -y poppler-utils
!pip install opencv-python numpy pdf2image img2pdf

In [ ]:
# ========================================================
# 1. INSTALACIÓN DE LIBRERÍAS (Si no están instaladas)
# ========================================================
import os
try:
    import cv2
    import img2pdf
    from pdf2image import convert_from_path
except ImportError:
    print("⏳ Instalando dependencias...")
    os.system('apt-get install -y poppler-utils')
    os.system('pip install opencv-python numpy pdf2image img2pdf')
    import cv2
    import img2pdf
    from pdf2image import convert_from_path

import numpy as np

# ========================================================
# 2. CONFIGURACIÓN DE RUTAS
# ========================================================
RUTA_ENTRADA = '/content/guio.pdf'
RUTA_SALIDA  = '/content/guio_ajustado_limpio.pdf'
# ========================================================

def limpiar_ajustado_final(input_path, output_path):
    if not os.path.exists(input_path):
        print(f"❌ No se encontró '{input_path}'. Súbelo a la carpeta de la izquierda.")
        return

    print("🚀 Iniciando limpieza de ruido y reconstrucción equilibrada...")
    paginas = convert_from_path(input_path, 300)
    archivos_temporales = []

    for i, pagina in enumerate(paginas):
        # A. Escala de grises
        img = cv2.cvtColor(np.array(pagina), cv2.COLOR_RGB2GRAY)

        # B. ELIMINACIÓN DE TEXTURA DE PAPEL (Denoising)
        # Filtramos el grano antes de binarizar para que no se convierta en puntos negros
        img = cv2.fastNlMeansDenoising(img, None, 10, 7, 21)

        # C. REALCE DE CONTRASTE CONTROLADO
        # Bajamos clipLimit a 1.5 (antes 2.5) para no "excitar" el ruido del fondo
        clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8,8))
        img = clahe.apply(img)

        # D. BINARIZACIÓN
        thresh = cv2.adaptiveThreshold(
            img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV, 15, 6
        )

        # E. ELIMINACIÓN DE RUIDO PEQUEÑO (Opening)
        # Borra los puntos negros que no llegan a ser letras
        kernel_clean = np.ones((2,2), np.uint8)
        img_clean = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_clean)

        # F. RECONSTRUCCIÓN DE LETRAS (Closing)
        # Ahora que está limpio, unimos los trazos de las letras
        k_h = cv2.getStructuringElement(cv2.MORPH_RECT, (2,1))
        k_v = cv2.getStructuringElement(cv2.MORPH_RECT, (1,2))
        img_rehecha = cv2.morphologyEx(img_clean, cv2.MORPH_CLOSE, k_h)
        img_rehecha = cv2.morphologyEx(img_rehecha, cv2.MORPH_CLOSE, k_v)

        # G. FILTRO DE ÁREA FINAL
        # Borramos cualquier mancha residual que haya quedado
        nlabels, labels, stats, centroids = cv2.connectedComponentsWithStats(img_rehecha, None, None, None, 8, cv2.CV_32S)

        img_final = np.zeros((labels.shape), dtype=np.uint8)
        for j in range(1, nlabels):
            area = stats[j, cv2.CC_STAT_AREA]
            # Subimos el área mínima a 12 píxeles para ser más estrictos con el ruido
            if 12 < area < 15000:
                img_final[labels == j] = 255

        # H. INVERSIÓN (Texto negro / Fondo blanco)
        img_final = cv2.bitwise_not(img_final)

        # Guardar temporal
        nombre_temp = f"temp_p{i}.png"
        cv2.imwrite(nombre_temp, img_final)
        archivos_temporales.append(nombre_temp)
        if (i+1) % 5 == 0: print(f"✅ {i+1} páginas procesadas...")

    # Reensamblar
    with open(output_path, "wb") as f:
        f.write(img2pdf.convert(archivos_temporales))

    for tmp in archivos_temporales: os.remove(tmp)
    print(f"\n✨ ¡PDF LISTO! Guardado como: {output_path}")

# Ejecución
limpiar_ajustado_final(RUTA_ENTRADA, RUTA_SALIDA)